# DwellSense / NestIQ: NLP Fake Review Detection Model
### Architecture: TF-IDF + Behavioral/Linguistic Feature Analytics + Logistic Regression
**Benchmark Dataset:** Deceptive Opinion Spam Corpus (Ott et al., Cornell University / ACL)

This notebook trains an explainable, high-accuracy machine learning model that evaluates both **review text** and **star rating (1.0 to 5.0)** to detect deceptive, troll, and spam reviews.

In [ ]:
# 1. Install & Import Libraries
import pandas as pd
import numpy as np
import re
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve

print('All libraries imported successfully!')

In [ ]:
# 2. Load the Dataset with Explicit Ratings
import os

dataset_filename = 'hostel_fake_reviews_dataset.csv'

if os.path.exists(dataset_filename):
    print(f'Loading existing local dataset: {dataset_filename}')
    df = pd.read_csv(dataset_filename)
else:
    print('Downloading benchmark dataset directly from Kaggle...')
    import kagglehub
    k_path = kagglehub.dataset_download('rtatman/deceptive-opinion-spam-corpus')
    csv_path = os.path.join(k_path, 'deceptive-opinion.csv')
    raw_df = pd.read_csv(csv_path)
    
    raw_df['label'] = raw_df['deceptive'].apply(lambda x: 0 if str(x).lower().strip() == 'truthful' else 1)
    np.random.seed(42)
    ratings = []
    for _, row in raw_df.iterrows():
        if str(row['polarity']).lower().strip() == 'positive':
            ratings.append(np.random.choice([5.0, 4.0], p=[0.8, 0.2]))
        else:
            ratings.append(np.random.choice([1.0, 2.0], p=[0.8, 0.2]))
    raw_df['rating'] = ratings
    
    balanced_3star = [
        ('The hostel is in a decent location and the staff is friendly, but the rooms are cramped and the wifi is unreliable.', 3.0, 0),
        ('Overall an average stay. The common area and washrooms are clean, but the food quality could definitely be improved.', 3.0, 0),
        ('Decent value for money. The air conditioning works well, but hot water was only available for limited hours in the morning.', 3.0, 0),
        ('The security and warden are good, but the street outside gets very noisy at night.', 3.0, 0),
        ('Adequate facilities for students on a budget. Rooms are basic but livable. Laundry machines are often out of order.', 3.0, 0),
        ('Fairly standard hostel experience. Friendly roommates and decent study room, though the internet speed drops during peak hours.', 3.0, 0),
    ] * 20
    b_df = pd.DataFrame(balanced_3star, columns=['text', 'rating', 'label'])
    df = pd.concat([raw_df[['text', 'rating', 'label']], b_df], ignore_index=True)
    df = df.rename(columns={'text': 'review_text'}).sample(frac=1, random_state=42).reset_index(drop=True)
    df.to_csv(dataset_filename, index=False)
    print(f'Saved formatted dataset with ratings as {dataset_filename}')

print(f'Total accommodation reviews: {len(df)}')
print('Columns in dataset:', df.columns.tolist())
print('\nClass distribution (0 = Genuine, 1 = Fake):')
print(df['label'].value_counts().rename({0: 'Genuine', 1: 'Fake'}))
print('\nRating distribution (1.0 to 5.0 stars):')
print(df['rating'].value_counts().sort_index())
print('\nFirst 3 rows with explicit ratings:')
print(df[['rating', 'label', 'review_text']].head(3))


In [ ]:
# 3. Behavioral & Linguistic Feature Extraction
POSITIVE_WORDS = {
    'good', 'great', 'excellent', 'amazing', 'love', 'nice', 'best', 'clean',
    'helpful', 'quiet', 'friendly', 'comfortable', 'recommend', 'spacious',
    'perfect', 'peaceful', 'wonderful', 'safe', 'delicious', 'value', 'superb',
    'pleasant', 'tidy', 'courteous', 'polite', 'cozy'
}
NEGATIVE_WORDS = {
    'bad', 'worst', 'terrible', 'horrible', 'dirty', 'noisy', 'poor', 'slow',
    'rude', 'broken', 'smell', 'bugs', 'disaster', 'scam', 'uncomfortable',
    'cramped', 'hated', 'hate', 'awful', 'overpriced', 'disgusting', 'filthy',
    'unacceptable', 'stinky', 'cockroach', 'freezing'
}

def extract_behavioral_features(texts, ratings):
    feats = []
    for text, rating in zip(texts, ratings):
        text_str = str(text) if pd.notnull(text) else ''
        words = re.findall(r'\b\w+\b', text_str.lower())
        num_words = len(words)
        char_len = len(text_str)
        
        p_cnt = sum(1 for w in words if w in POSITIVE_WORDS)
        n_cnt = sum(1 for w in words if w in NEGATIVE_WORDS)
        tot = p_cnt + n_cnt
        sentiment = (p_cnt - n_cnt) / tot if tot > 0 else 0.0
        
        # Rating normalized to [-1.0, +1.0]: 1.0 -> -1.0, 3.0 -> 0.0, 5.0 -> +1.0
        norm_rating = (float(rating) - 3.0) / 2.0
        discrepancy = abs(norm_rating - sentiment)
        
        caps_ratio = sum(1 for c in text_str if c.isupper()) / max(1, char_len)
        punct_density = (text_str.count('!') + text_str.count('?')) / max(1, num_words)
        elongation = 1.0 if re.search(r'(.)\1{2,}', text_str.lower()) else 0.0
        
        feats.append([
            float(rating), sentiment, discrepancy, float(num_words),
            float(char_len), caps_ratio, punct_density, elongation
        ])
    return np.array(feats)

print('Feature engineering functions defined successfully!')

In [ ]:
# 4. Model Training: TF-IDF + Logistic Regression
X_train_text, X_test_text, r_train, r_test, y_train, y_test = train_test_split(
    df['review_text'].values, df['rating'].values, df['label'].values,
    test_size=0.2, random_state=42, stratify=df['label'].values
)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words='english', sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Behavioral Scaler
feat_train = extract_behavioral_features(X_train_text, r_train)
feat_test = extract_behavioral_features(X_test_text, r_test)
scaler = StandardScaler()
feat_train_s = scaler.fit_transform(feat_train)
feat_test_s = scaler.transform(feat_test)

# Combine text features + behavioral features
X_train_full = sp.hstack([X_train_tfidf, feat_train_s], format='csr')
X_test_full = sp.hstack([X_test_tfidf, feat_test_s], format='csr')

# Train Logistic Regression Classifier
clf = LogisticRegression(C=5.0, max_iter=1000, random_state=42)
clf.fit(X_train_full, y_train)

# Predict & Evaluate
y_pred = clf.predict(X_test_full)
y_prob = clf.predict_proba(X_test_full)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f'Test Accuracy : {acc * 100:.2f}%')
print(f'ROC-AUC Score : {auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Genuine (0)', 'Fake (1)']))

In [ ]:
# 5. Performance Visualizations (Confusion Matrix & ROC Curve)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Genuine', 'Fake'], yticklabels=['Genuine', 'Fake'], ax=axes[0])
axes[0].set_title('Confusion Matrix (Test Split)')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Top Learned Deceptive vs. Genuine Words (Explainability)
feature_names = list(tfidf.get_feature_names_out()) + [
    'FEAT_rating', 'FEAT_sentiment', 'FEAT_discrepancy',
    'FEAT_word_count', 'FEAT_char_len', 'FEAT_caps_ratio',
    'FEAT_punct_density', 'FEAT_elongation'
]
coefs = clf.coef_[0]

top_fake_idx = np.argsort(coefs)[-12:]
top_genuine_idx = np.argsort(coefs)[:12]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.barh([feature_names[i] for i in top_fake_idx], coefs[top_fake_idx], color='salmon')
plt.title('Top Deceptive/Fake Feature Weights (+)')
plt.xlabel('Coefficient Weight')

plt.subplot(1, 2, 2)
plt.barh([feature_names[i] for i in top_genuine_idx], coefs[top_genuine_idx], color='lightgreen')
plt.title('Top Genuine Feature Weights (-)')
plt.xlabel('Coefficient Weight')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Interactive Review Prediction Widget
def evaluate_review(comment: str, rating: float):
    feat_arr = extract_behavioral_features([comment], [rating])
    feat_scaled = scaler.transform(feat_arr)
    text_tfidf = tfidf.transform([comment])
    X_in = sp.hstack([text_tfidf, feat_scaled], format='csr')
    
    prob = float(clf.predict_proba(X_in)[0][1])
    
    # Explainable reasons
    reasons = []
    words = re.findall(r'\b\w+\b', comment.lower())
    norm_r = (float(rating) - 3.0) / 2.0
    p_cnt = sum(1 for w in words if w in POSITIVE_WORDS)
    n_cnt = sum(1 for w in words if w in NEGATIVE_WORDS)
    tot = p_cnt + n_cnt
    sent = (p_cnt - n_cnt) / tot if tot > 0 else 0.0
    
    if abs(norm_r - sent) > 1.2:
        reasons.append(f'Rating-Sentiment contradiction: {rating} stars but opposite text sentiment.')
    if len(words) < 5 and (rating >= 4.5 or rating <= 1.5):
        reasons.append('Extreme rating with minimal text detail (< 5 words).')
    if re.search(r'(.)\1{2,}', comment.lower()):
        reasons.append('Character repetition spamming detected.')
        
    is_fake = bool(prob >= 0.5 or (len(reasons) > 0 and prob >= 0.30))
    
    print('='*55)
    print(f'Review Text  : "{comment}"')
    print(f'Star Rating  : {rating} / 5.0')
    print(f'Prediction   : {"🚨 SUSPICIOUS / FAKE" if is_fake else "✅ GENUINE"}')
    print(f'Fake Probability: {prob:.4f} ({prob*100:.1f}%)')
    if reasons:
        print('Flag Reasons :')
        for r in reasons:
            print(f'  - {r}')
    print('='*55)

# Try custom examples
evaluate_review('The hostel is clean and staff is polite, but the internet drops in the evening.', 3.0)
evaluate_review('O.M.G best hostel ever greaaatest place visit now!', 5.0)
evaluate_review('Worst disaster dirty room broken bathroom!', 5.0)

In [ ]:
# 8. Export Model for FastAPI / Flutter Backend
model_bundle = {
    'vectorizer': tfidf,
    'scaler': scaler,
    'classifier': clf,
    'accuracy': float(acc),
    'auc': float(auc)
}
joblib.dump(model_bundle, 'fake_review_model.joblib')
print('Model saved as fake_review_model.joblib successfully!')
# Uncomment to download directly in Colab:
# from google.colab import files
# files.download('fake_review_model.joblib')